In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
# Tokenizer
with open("mixed_ipa_corpus.txt", "r", encoding="utf-8") as f:
    text = f.read()

ipa_list = sorted(list(set(text)))
print(ipa_list)


id2ipa = {i: char for i, char in enumerate(ipa_list)}
ipa2id = {char: i for i, char in enumerate(ipa_list)}

print(ipa2id)
print(id2ipa)

['\n', ',', '.', ':', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', 'æ', 'ç', 'ð', 'ø', 'ħ', 'ŋ', 'ƒ', 'ɑ', 'ɒ', 'ɕ', 'ɛ', 'ɟ', 'ɡ', 'ɣ', 'ɦ', 'ɨ', 'ɯ', 'ɰ', 'ɲ', 'ɴ', 'ɸ', 'ɾ', 'ʂ', 'ʃ', 'ʋ', 'ʑ', 'ʒ', 'ʔ', 'ʕ', 'ʲ', 'ː', 'ˤ', '̪', '͡', 'θ', 'ћ', 'ء', 'ة', 'ـ', 'ى', 'ٓ', 'ٔ', 'ٕ', 'ٹ', 'ھ', 'ᵝ', 'ḱ']
{'\n': 0, ',': 1, '.': 2, ':': 3, 'a': 4, 'b': 5, 'c': 6, 'd': 7, 'e': 8, 'f': 9, 'g': 10, 'h': 11, 'i': 12, 'j': 13, 'k': 14, 'l': 15, 'm': 16, 'n': 17, 'o': 18, 'p': 19, 'q': 20, 'r': 21, 's': 22, 't': 23, 'u': 24, 'v': 25, 'w': 26, 'x': 27, 'y': 28, 'z': 29, 'æ': 30, 'ç': 31, 'ð': 32, 'ø': 33, 'ħ': 34, 'ŋ': 35, 'ƒ': 36, 'ɑ': 37, 'ɒ': 38, 'ɕ': 39, 'ɛ': 40, 'ɟ': 41, 'ɡ': 42, 'ɣ': 43, 'ɦ': 44, 'ɨ': 45, 'ɯ': 46, 'ɰ': 47, 'ɲ': 48, 'ɴ': 49, 'ɸ': 50, 'ɾ': 51, 'ʂ': 52, 'ʃ': 53, 'ʋ': 54, 'ʑ': 55, 'ʒ': 56, 'ʔ': 57, 'ʕ': 58, 'ʲ': 59, 'ː': 60, 'ˤ': 61, '̪': 62, '͡': 63, 'θ': 64, 'ћ': 65, 'ء': 66, 'ة': 67, 'ـ': 68, 

In [3]:
data = torch.tensor([ipa2id[c] for c in text], dtype=torch.long)
print(f"total char num {len(data)}")
print(data[:100])

total char num 208467
tensor([10,  8, 49, 29, 18, 46,  0, 10,  8, 49, 22, 18, 14, 46,  0, 14,  8, 49,
        23, 39, 12,  0, 10,  8, 49, 23, 39, 12,  0, 10,  8, 49, 23,  8, 12,  0,
        10,  8, 49, 23,  8, 49,  0, 10,  8, 49, 23,  8, 49,  0, 10,  8, 49,  5,
         4, 14, 46,  0, 10,  8, 49,  5, 46, 49,  0, 10,  8, 49, 16, 12, 23, 22,
        46,  0, 14,  8, 49, 16,  8, 12,  0, 14,  8, 49, 13,  4, 14, 46,  0, 10,
         8, 49, 13, 46,  0, 14,  8, 49, 13, 18])


In [4]:
n = int(0.9 * len(data)) # 90%for Training、10% for testing
train_data = data[:n]
val_data = data[n:]

In [5]:
torch.manual_seed(1337) # consting seed

batch_size = 32  # batch size for 1 shot
block_size = 8   # max len of a word

def get_batch(split):
    # "train" for training data
    # "test" for testing data
    data_source = train_data if split == 'train' else val_data

    # getting random starting point
    ix = torch.randint(len(data_source) - block_size, (batch_size,))

    # x: input、y: the char that next to x
    x = torch.stack([data_source[i : i+block_size] for i in ix])
    y = torch.stack([data_source[i+1 : i+block_size+1] for i in ix])

    return x, y

# 1 batch test
xb, yb = get_batch('train')
print("---")
print("inputting x data(batch_size, block_size):", xb.shape)
print("Answer y data(batch_size, block_size):", yb.shape)

---
inputting x data(batch_size, block_size): torch.Size([32, 8])
Answer y data(batch_size, block_size): torch.Size([32, 8])


In [6]:
# 辞書のサイズ（発見されたIPAの種類数）
vocab_size = len(ipa2id)
n_embd = 64 # 次元数（脳の神経の太さ。今回は64で十分です）

class CharTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        # 1. 文字IDを、AIが計算しやすい64次元のベクトル（意味の塊）に変換する層
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)

        # 2. 文字の「位置（何文字目か）」を教える層（これがないと順番が理解できない）
        self.position_embedding_table = nn.Embedding(block_size, n_embd)

        # 3. Transformerのメインエンジン（今回はシンプルに2層）
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=n_embd,
            nhead=4,
            dim_feedforward=128,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)

        # 4. 最後にまた「IDごとの確率」に戻すための出力層
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape # B:バッチサイズ, T:文字の長さ

        # 文字の埋め込みベクトル ＋ 位置情報のベクトル を足し合わせる
        tok_emb = self.token_embedding_table(idx)
        pos = torch.arange(0, T, dtype=torch.long)
        pos_emb = self.position_embedding_table(pos)
        x = tok_emb + pos_emb

        # 未来の文字をカンニングできないようにするマスク（超重要！）
        mask = nn.Transformer.generate_square_subsequent_mask(T)

        # Transformerブロックに流し込む
        x = self.transformer(x, mask=mask, is_causal=True)

        # 各トークンの「次の文字」の予測スコア（logits）
        logits = self.lm_head(x)

        # targets（正解データ）が渡された場合は、答え合わせをして誤差（Loss）を計算
        if targets is None:
            loss = None
        else:
            # PyTorchのcross_entropy関数の仕様に合わせて形状を平たくする
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

# モデルをインスタンス化
model = CharTransformer()

# 先ほど作成したテスト用のミニバッチ(xb, yb)を流し込んでみる
logits, loss = model(xb, yb)

print("出力データの形状 (B*T, 辞書サイズ):", logits.shape)
print(f"初期状態の誤差 (Loss): {loss.item():.4f}")

出力データの形状 (B*T, 辞書サイズ): torch.Size([256, 77])
初期状態の誤差 (Loss): 4.5469


In [7]:
# =========================================
# 5. 最適化アルゴリズム（Optimizer）のセット
# =========================================
# AIの学習において最も優秀で定番な「AdamW」を使います
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

max_iters = 3000      # 学習のループ回数
eval_interval = 300   # Lossを表示する間隔

for iter in range(max_iters):
    # 1. ミニバッチ（学習データ）を取得
    xb, yb = get_batch('train')

    # 2. 順伝播：モデルに予測させ、正解との誤差（Loss）を計算
    logits, loss = model(xb, yb)

    # 3. 逆伝播：誤差をもとに、脳内のパラメーターをどう修正すべきか計算
    optimizer.zero_grad(set_to_none=True) # 前回の計算結果（勾配）をリセット
    loss.backward()                       # 誤差を逆伝播

    # 4. パラメーターの更新：実際に脳のネットワークを少しだけ賢くする
    optimizer.step()

    # 一定間隔で現在のLossを表示して進捗を確認
    if iter % eval_interval == 0:
        print(f"Step {iter}: Loss = {loss.item():.4f}")

print(f"🎉 学習完了！最終Loss: {loss.item():.4f}")

Step 0: Loss = 4.5143
Step 300: Loss = 2.7515
Step 600: Loss = 2.5369
Step 900: Loss = 2.3877
Step 1200: Loss = 2.4303
Step 1500: Loss = 2.4617
Step 1800: Loss = 2.3981
Step 2100: Loss = 2.4068
Step 2400: Loss = 2.3606
Step 2700: Loss = 2.3614
🎉 学習完了！最終Loss: 2.3971


In [9]:
# =========================================
# 6. 未知の人工言語（IPA）の生成！
# =========================================
# 最初はID 0のトークン（おそらく改行など）から生成をスタートさせます
context = torch.zeros((1, 1), dtype=torch.long)

# 何文字のIPAを生成するか
max_new_tokens = 300
temperature = 0.5

for _ in range(max_new_tokens):
    # 現在の文脈（直近の block_size 文字）だけを切り出して入力
    idx_cond = context[:, -block_size:]

    # モデルに予測させる
    logits, _ = model(idx_cond)

    logits = logits[:, -1, :] / temperature
    probs = F.softmax(logits, dim=-1)

    # 確率に基づいて次の文字を1つサンプリング（ガチャを回す）
    idx_next = torch.multinomial(probs, num_samples=1)

    # 予測した文字を文脈に追加して、次のループへ
    context = torch.cat((context, idx_next), dim=1)

# 生成されたIDの配列（テンソル）を普通のPythonのリストに変換
generated_ids = context[0].tolist()

# IDをIPA記号（テキスト）に戻す
generated_text = "".join([id2ipa[i] for i in generated_ids])

print(generated_text)


lʔa
alsˤiː
alʔaaː
aٔqd͡ʒd͡ʒ
alqtsˤr
alm
alθaːaː
alðkaːm
aٔtaːk
alqlf
slaː
fs
aٔnd͡ʒaː
aٔfbuːl
alʔand͡ʒ
almaːqaː
ʃaːrt
tod͡ʒɨl
votvonʲenʲenʲenʲirʲim
nʲerasʲa
nʲet͡ɕʲatʲ
ʒɨx
kavalʲːla
zantaka
saɾita
kaɾazabɯ
ɕaɴ
kakɯkɯ
koɯɕikɯ
ɸɯ
kaɴtaɾɯ
teɴ
kai
ɕitai
kateɴ
koɴkaɴ
katɕiɴ
taɴdʑo
gɯkai
daɴta
bɯɴtoɯ
ɕiɴd


In [19]:
# generate based on user's first input

input_text = "kaɾazabɯ\n"
model.eval()

temperature=0.5
max_len=100
new_line_count = 0

# to ipa to id
input_ids = []
for _ in input_text:
  input_ids.append(ipa2id[_])
context = torch.tensor([input_ids], dtype=torch.long)
generated_ids = []

with torch.no_grad():
    while True:
        # 直近の block_size 分だけをクロップ
        idx_cond = context[:, -block_size:]
        logits, _ = model(idx_cond)

        # Temperatureを適用して最後のトークンの予測確率を計算
        logits = logits[:, -1, :] / temperature
        probs = F.softmax(logits, dim=-1)

        # ガチャ（サンプリング）
        idx_next = torch.multinomial(probs, num_samples=1)
        next_char = id2ipa[idx_next.item()]

        # if it counts max_len "\n" break
        if "\n" in next_char or " " in next_char:
            new_line_count += 1
            if new_line_count >= max_len:
                break


        generated_ids.append(idx_next.item())
        context = torch.cat((context, idx_next), dim=1)

next_word = "".join([id2ipa[i] for i in generated_ids])
print(next_word)
with open("new_language.txt", "w", encoding="utf-8") as f:
    f.write(next_word)



kaɴ
koɯkaɴ
saɴ
kaɴsaɴ
kaɴkai
kaɴdʑi
kaɴkaɴ
koɯɕikaɴ
tɕiɴto
oɯ
koɯ
kokɯɴ
keɴdʑi
kaɴseɴ
sɯkiɴ
gaiɾiɴ
tai
ɕimakɯ
seɴkoɯ
gaika
kaitɕi
seɴkaɴ
ɕoɯkaɴ
tɕi
boɯɕi
kiɴ
koɯiɴka
kaɴɕikaɴ
toɴ
hoɯkeɴ
toɯki
tsɯka
daki
aٔmaːn
alqd͡ʒ
alql
mb
alhaː
aٔriːd
alʔa
saːħdb
aٔaːiːh
alʃaː
almʃaːn
maːn
uːaːk
m
bnsiː
alʔan
iːd͡ʒuːl
aٔmsˤuː
alniːt
almqd
altqaː
almiːdaː
almaː
ald͡ʒd͡ʒuːt
altˤaː
almd͡ʒiːq
iːstf
almnaːqd͡ʒiːة
almd͡ʒ
aٔiːd
aٔntˤiːb
alʔaʕ
sraːd
almriː
alʔauːd
alʕuːsaːl
mltˤl
aٔmm
alħlʔaniː
sˤaːm
albɒzt
mɒnoː
roːjɒroː
ɒt͡sɛnː
miːtɛk
nɛtːɛtˤɛn
tɛʃɛʃiːdɛk
ɛɟɛn
initɛk
mɒnɒm
ɒjomɒto
sɒl
bɒntɒ
mɒm
oːtɒt͡ɒnaː
ɒbːɛn
min
pːonata
toke
podʲit͡ɕʲetʲilo
prozadʲe
t͡ɕʲe
prodoka
posart͡ɕʲetʲ
dovovʲelʲitʲ
prʲerʲitʲ
vovova
